# Perceptual evaluation based on GMMs in the feature space

We replace the MW metric used in 
[Luzi et al., 2023](https://openaccess.thecvf.com/content/WACV2023/papers/Luzi_Evaluating_Generative_Networks_Using_Gaussian_Mixtures_of_Image_Features_WACV_2023_paper.pdf)
with our DSMW/MSW metric.

In [6]:
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
import os
from torchvision.models import inception_v3
from scipy.linalg import sqrtm
import GMM_utils as GMM
import sliced_mw as SMW
import warnings
import matplotlib.pyplot as plt
import ot
import scipy.stats as sps
import scipy.linalg as spl
from scipy.optimize import linprog

np.random.seed(42)
torch.manual_seed(42)

plt.rcParams.update({'font.size': 22})

reg_cov = 1e-2
warnings.filterwarnings("ignore")
torch.cuda.empty_cache()


def GaussianW2(m0,m1,Sigma0,Sigma1):
    # Wasserstein between Gaussians
    # source: https://github.com/judelo
    Sigma00  = spl.sqrtm(Sigma0)
    Sigma010 = spl.sqrtm(Sigma00@Sigma1@Sigma00)
    d = np.linalg.norm(m0-m1)**2
    d =+np.trace(Sigma0+Sigma1-2*Sigma010)
    return d

def MW2(pi_0,pi_1,mu_0,mu_1,Sigma0_arr,Sigma1_arr):
    # Return the MW dist
    # source: https://github.com/judelo
    K0 = mu_0.shape[0]
    K1 = mu_1.shape[0]
    d  = mu_0.shape[1]
    Sigma0_arr = Sigma0_arr.reshape(K0,d,d)
    Sigma1_arr = Sigma1_arr.reshape(K1,d,d)
    M  = np.zeros((K0,K1))
    
    # Pairwise Wasserstein distance matrix between all Gaussians
    for k in range(K0):
        for l in range(K1):
            M[k,l]  = GaussianW2(mu_0[k,:],mu_1[l,:],Sigma0_arr[k,:,:],Sigma1_arr[l,:,:])
    # Compute OT distance
    wstar     = ot.emd(pi_0,pi_1,M)      
    dist   = np.sum(wstar*M)
    return dist

def calc_MW_org(gmm1, gmm2):
    return MW2(gmm1.weights.numpy(), gmm2.weights.numpy(),
                        gmm1.means.numpy(), gmm2.means.numpy(),
                        gmm1.covariances.numpy(), gmm2.covariances.numpy())

def add_noise(images, noise_level):
    # Gaussian noise
    noise = torch.randn_like(images) * noise_level
    noisy_images = torch.clamp(images + noise, 0, 1)  # Keep values in valid range
    return noisy_images

def add_salt_and_pepper_noise(images, noise_level):
    """
    Applies salt and pepper noise to a batch of images.
    
    Args:
        images (torch.Tensor): Tensor of shape (N, C, H, W).
        noise_level (float): Fraction of pixels to alter.
        
    Returns:
        torch.Tensor: Noisy images.
    """
    rand_tensor = torch.rand_like(images)
    noisy_images = images.clone()
    noisy_images[rand_tensor < (noise_level / 2)] = 0.0  # Pepper
    noisy_images[rand_tensor > 1 - (noise_level / 2)] = 1.0  # Salt
    return noisy_images

def apply_gaussian_blur(images, sigma):
    """
    Applies Gaussian blur to a batch of images.
    
    Args:
        images (torch.Tensor): Tensor of shape (N, C, H, W).
        sigma (float): Standard deviation for Gaussian kernel.
    
    Returns:
        torch.Tensor: Blurred images.
    """
    gaussian_blur = transforms.GaussianBlur(kernel_size=3, sigma=sigma)
    # Since GaussianBlur may not support batched tensors directly,
    # apply it to each image individually.
    blurred_images = torch.stack([gaussian_blur(img) for img in images])
    return blurred_images

def apply_distortion(images, distortion_type, distortion_level):
    """
    Applies the selected distortion to the images.
    
    Args:
        images (torch.Tensor): Tensor of shape (N, C, H, W).
        distortion_type (str): Type of distortion ("Gaussian", "SP", or "Blur").
        distortion_level (float): Level of distortion to apply.
        
    Returns:
        torch.Tensor: Distorted images.
    """
    dt = distortion_type
    if dt == "Gaussian":
        return add_noise(images, distortion_level)
    elif dt == "SP":
        return add_salt_and_pepper_noise(images, distortion_level)
    elif dt == "Blur":
        return apply_gaussian_blur(images, distortion_level)
    else:
        raise ValueError(f"Unknown distortion type: {distortion_type}")

def compute_embeddings_batchwise(images, model, device, batch_size=50):
    """
    Computes embeddings in batches to avoid memory issues.
    
    Args:
        images (torch.Tensor): Tensor of shape (N, C, H, W).
        model (torch.nn.Module): Inception model without the final fc layer.
        device (str): 'cuda' or 'cpu'.
        batch_size (int): Number of images per batch.
        
    Returns:
        np.ndarray: Embeddings of shape (N, embedding_dim).
    """
    model.eval()
    resizer = transforms.Resize(299)
    embeddings_list = []
    num_images = images.shape[0]
    
    with torch.no_grad():
        for i in range(0, num_images, batch_size):
            batch = images[i:i+batch_size].to(device)
            # Resize each image to 299x299 (expected by Inception v3)
            batch = resizer(batch)
            emb = model(batch).cpu().numpy()
            embeddings_list.append(emb)
    return np.concatenate(embeddings_list, axis=0)

def calculate_fid_from_embeddings(embeddings1, embeddings2):
    """
    Calculate the Frechet Inception Distance (FID) between two sets of embeddings.
    
    Args:
        embeddings1 (np.ndarray): Embeddings of set 1, shape (N, D).
        embeddings2 (np.ndarray): Embeddings of set 2, shape (N, D).
        
    Returns:
        float: The FID score.
    """
    mu1 = np.mean(embeddings1, axis=0)
    mu2 = np.mean(embeddings2, axis=0)
    sigma1 = np.cov(embeddings1, rowvar=False)
    sigma2 = np.cov(embeddings2, rowvar=False)
    
    # Add a small identity matrix to prevent numerical errors
    epsilon = reg_cov
    sigma1 += epsilon * np.eye(sigma1.shape[0])
    sigma2 += epsilon * np.eye(sigma2.shape[0])
    
    diff = mu1 - mu2
    diff_squared = diff.dot(diff)
    
    # Compute the square root of the product of covariance matrices
    covmean = sqrtm(sigma1.dot(sigma2))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    
    fid = diff_squared + np.trace(sigma1 + sigma2 - 2 * covmean)
    return fid

def compute_fid(real_images, noise_levels, embeddings_folder="embeddings", batch_size=50, 
                type="FID", distortion_type="Gaussian", save=True):
    """
    Computes FID scores between real images and distorted versions by
    extracting Inception embeddings batchwise and saving them.
    
    Args:
        real_images (torch.Tensor): Tensor of shape (N, C, H, W) for real images.
        noise_levels (list): List of distortion levels to add.
        embeddings_folder (str): Folder to save embeddings.
        batch_size (int): Batch size for embedding extraction.
        type (str): Either "FID" or "GMM_SMWS".
        distortion_type (str): Type of distortion ("Gaussian", "SP", or "Blur").
        
    Returns:
        dict: Mapping noise_level -> FID score.
    """
    os.makedirs(embeddings_folder, exist_ok=True)
    plots_folder = "perception_plots"
    os.makedirs(plots_folder, exist_ok=True)
    fid_scores = {}
    dt = distortion_type

    device = "cuda" if (torch.cuda.is_available() and type == "GMM_SMSW" and type != "GMM_MW") else "cpu"
    inception = inception_v3(pretrained=True, transform_input=True).to(device)
    # Remove the classification head by replacing it with an identity function
    inception.fc = torch.nn.Identity()
    
    # Compute embeddings for real images batchwise and save them.
    savefile = "real_embeddings.npy"
    savefile = os.path.join(embeddings_folder, savefile)
    try:
        real_embeddings = np.load(savefile)
        print("Loaded Real Embedding")
    except:
        real_embeddings = compute_embeddings_batchwise(real_images, inception, device, batch_size=batch_size)
        np.save(savefile, real_embeddings)
    
    if "GMM" in type:
        torch_real_embeddings = torch.tensor(real_embeddings).to(device)
        real_gmm = GMM.FittedGaussianMixtureModel(torch_real_embeddings, 5, device=device, reg_cov=reg_cov)
    
    for noise_level in noise_levels:
        savefile_raw = f"fake_embeddings_{dt}_{noise_level}.npy"
        savefile = os.path.join(embeddings_folder, savefile_raw)
        try:
            fake_embeddings = np.load(savefile)
            print("Loaded ", savefile_raw)
        except:
            distorted_images = apply_distortion(real_images, distortion_type, noise_level)
            fake_embeddings = compute_embeddings_batchwise(distorted_images, inception, device, batch_size=batch_size)
            np.save(savefile, fake_embeddings)
        if "GMM" in type:
            torch_fake_embeddings = torch.tensor(fake_embeddings).to(device)
            fake_gmm = GMM.FittedGaussianMixtureModel(torch_fake_embeddings, 10, device=device)
        

        pnum = 10000
        t0 = time.time()
        if type == "FID":
            fid = calculate_fid_from_embeddings(real_embeddings, fake_embeddings)
        elif type == "GMM_SMSW":
            fid = SMW.calc_SMSW(fake_gmm, real_gmm, pnum=pnum).item()
        elif type == "GMM_MSW":
            fid = SMW.calc_MSW(fake_gmm, real_gmm, pnum=pnum).item()
        elif type == "GMM_MW":
            fid = np.array(calc_MW_org(fake_gmm, real_gmm))
        elif type == "GMM_SMW":
            fid = SMW.calc_test_SMW(fake_gmm, real_gmm, pnum=pnum).item()
        fid_scores[noise_level] = fid
        print(f"{type} for {distortion_type} distortion at level {noise_level}: {fid}")
        print("Seconds: ", -(t0 - time.time()))
        del fake_embeddings
        torch.cuda.empty_cache()
    
    # Plot the FID-type values vs distortion level
    levels = sorted(fid_scores.keys())
    scores = [fid_scores[lvl] for lvl in levels]
    if save:
        plt.figure(figsize=(8,6))
        plt.plot(levels, scores, marker='o')
        plt.xlabel("Distortion Level")
        plt.grid(True)
        plot_filename = os.path.join(plots_folder, f"{type}_{distortion_type}_plot.png")
        plt.savefig(plot_filename)
        plt.show()
        print(f"Plot saved to {plot_filename}")

    return fid_scores

if __name__ == "__main__":
    # Load CIFAR-10 images
    transform = transforms.Compose([transforms.ToTensor()])
    dataset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
    real_images = torch.stack([dataset[i][0] for i in range(1000)])  # Adjust number of images as needed

    noise_levels = np.linspace(.5, 1.5, 10)
    dt = "Blur"
    fid_scores = compute_fid(real_images, noise_levels, batch_size=10, type="GMM_MW", distortion_type=dt)
    fid_scores = compute_fid(real_images, noise_levels, batch_size=10, type="GMM_MSW", distortion_type=dt)
    ffid_scores = compute_fid(real_images, noise_levels, batch_size=10, type="GMM_SMSW", distortion_type=dt)
    fid_scores = compute_fid(real_images, noise_levels, batch_size=10, type="FID", distortion_type=dt)



    noise_levels = np.linspace(.05, .3, 10)
    dt = "SP"
    fid_scores = compute_fid(real_images, noise_levels, batch_size=10, type="GMM_MW", distortion_type=dt)
    fid_scores = compute_fid(real_images, noise_levels, batch_size=10, type="GMM_MSW", distortion_type=dt)
    fid_scores = compute_fid(real_images, noise_levels, batch_size=10, type="GMM_SMSW", distortion_type=dt)
    fid_scores = compute_fid(real_images, noise_levels, batch_size=10, type="FID", distortion_type=dt)

    noise_levels = np.linspace(.01, .2, 10)
    dt = "Gaussian"
    fid_scores = compute_fid(real_images, noise_levels, batch_size=10, type="GMM_MW", distortion_type=dt)
    fid_scores = compute_fid(real_images, noise_levels, batch_size=10, type="GMM_MSW", distortion_type=dt)
    fid_scores = compute_fid(real_images, noise_levels, batch_size=10, type="GMM_SMSW", distortion_type=dt)
    fid_scores = compute_fid(real_images, noise_levels, batch_size=10, type="FID", distortion_type=dt)

Files already downloaded and verified
